# 中证800 V88：V46 股票池边界与流动性全A适配实验

本实验只改变股票池，冻结 V46 特征、raw-L2、fixed120、训练边界与 Top8 板块约束。比较：

- `hs300`：沪深300可交易成分股
- `csi800`：中证800可交易成分股（主线基准）
- `liquid_all_a_1500`：沪深A股中排除ST、停牌、上市不足180天后，20日平均成交额最高的1500只

不同股票池均用内部未来收益 Top2.5% 定义真实正例，避免固定 Top20 随股票池规模产生机械偏差。V46 因子保持不变；覆盖率、分布漂移和选股画像只用于适配审计，不进入模型。

数据重建按月缓存，支持断点续跑。默认运行 5 个年度 OOS folds × 3 seeds × 3 universes，共45个模型。


## 0. 导入、目录与进度条


In [ ]:
import os
import gc
import json
import warnings
import builtins as _bi
import datetime as _dt
from pathlib import Path

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception:
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v88_universe_boundary_outputs"
CACHE_DIR = PROJECT_DIR / "csi800_ml_v88_universe_monthly_cache"
FIGURE_DIR = OUT_DIR / "figures"
for _path in [OUT_DIR, CACHE_DIR, FIGURE_DIR]:
    _path.mkdir(parents=True, exist_ok=True)
RUN_TIMESTAMP = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
print("OUT_DIR:", OUT_DIR)


## 1. 冻结配置与预注册口径


In [ ]:
# Data rebuild is required on first run. Monthly cache files make interrupted runs resumable.
REBUILD_DATA = True
REBUILD_FORCE_MONTHS = False
DATA_PATH_OVERRIDE = None
REBUILD_START = "2019-01-01"
REBUILD_END_FOR_LABEL = None
REBUILD_END_RESOLVED = REBUILD_END_FOR_LABEL or _dt.datetime.now().strftime("%Y-%m-%d")
PANEL_PATH = PROJECT_DIR / ("v88_v46_universe_panel_%s_%s.csv" % (
    REBUILD_START.replace("-", ""), REBUILD_END_RESOLVED.replace("-", "")))

LIQUID_ALL_A_SIZE = 1500
MIN_LISTING_DAYS = 180
PRICE_CHUNK_SIZE = 180
LIQUIDITY_SCAN_CHUNK_SIZE = 350
FACTOR_CHUNK_SIZE = 18

UNIVERSES = [
    {"universe": "hs300", "membership_col": "in_hs300", "description": "tradable HS300"},
    {"universe": "csi800", "membership_col": "in_csi800", "description": "tradable CSI800 baseline"},
    {"universe": "liquid_all_a_1500", "membership_col": "in_liquid_all_a_1500", "description": "top1500 by trailing 20d money"},
]
UNIVERSE_NAMES = [x["universe"] for x in UNIVERSES]
BASELINE_UNIVERSE = "csi800"

STOCK_COL = "stock"
DATE_COL = "rebalance_date"
NEXT_DATE_COL = "next_date"
TARGET_COL = "alpha_1m"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"

FOLD_SPECS = [
    {"fold_id": "oos_2022", "cutoff": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "oos_2023", "cutoff": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "oos_2024", "cutoff": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "oos_2025", "cutoff": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "oos_2026", "cutoff": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]
SEEDS = [17, 42, 101]
FIXED_ITER = 120
CORR_THRESHOLD = 0.70
MIN_TRAIN_MONTHS = 30
TRUE_TOP_FRACTION = 0.025
PRED_TOP_K = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
NUM_THREADS = 4

BASE_PARAMS_FF10 = {
    "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
    "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200,
    "feature_fraction": 1.0, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 0.3, "verbose": -1,
}

# Smoke mode still keeps all universes; it only narrows folds and seeds.
SMOKE_TEST = False
SMOKE_MAX_FOLDS = 1
SMOKE_SEEDS = [42]
if SMOKE_TEST:
    FOLD_SPECS = FOLD_SPECS[:SMOKE_MAX_FOLDS]
    SEEDS = list(SMOKE_SEEDS)

print("panel:", PANEL_PATH)
print("universes:", UNIVERSE_NAMES)
print("models planned:", len(UNIVERSES) * len(FOLD_SPECS) * len(SEEDS))


## 2. V46 特征、排名指标与兼容工具


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)
AUDIT_ONLY_FACTORS = ["size"]
FETCH_JQFACTORS = unique_keep_order(BASE_FACTOR_COLS + AUDIT_ONLY_FACTORS)


def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_rank_ic(a, b):
    d = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 3 or d["a"].nunique() < 2 or d["b"].nunique() < 2:
        return np.nan
    return d["a"].rank(method="average").corr(d["b"].rank(method="average"))


def binary_rank_auc(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    positive = d["y"] > 0
    n_pos = int(positive.sum())
    n_neg = int(len(d) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = d["score"].rank(method="average")
    return (float(ranks[positive].sum()) - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg)


def binary_ndcg_at_k(y_true, scores, k):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "score": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("score", ascending=False)
    k_eff = _bi.min(int(k), len(d))
    if k_eff <= 0:
        return np.nan
    gains = (d["y"].head(k_eff).values > 0).astype(float)
    dcg = float((gains / np.log2(np.arange(k_eff, dtype=float) + 2.0)).sum())
    ideal_hits = _bi.min(k_eff, int((d["y"] > 0).sum()))
    if ideal_hits <= 0:
        return np.nan
    idcg = float((np.ones(ideal_hits) / np.log2(np.arange(ideal_hits, dtype=float) + 2.0)).sum())
    return dcg / idcg if idcg > 0 else np.nan


def stock_board(stock):
    code_value = str(stock).split(".")[0]
    if code_value.startswith(("300", "301")):
        return "chinext"
    if code_value.startswith(("688", "689")):
        return "star"
    return "other"


def board_capped_indices(sorted_df, k):
    selected = []
    counts = {"chinext": 0, "star": 0}
    for idx, row in sorted_df.iterrows():
        board = stock_board(row[STOCK_COL])
        if board in BOARD_CAPS and counts[board] >= BOARD_CAPS[board]:
            continue
        selected.append(idx)
        if board in counts:
            counts[board] += 1
        if len(selected) >= k:
            break
    return selected


def safe_mean(values):
    s = pd.Series(list(values), dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.mean()) if len(s) else np.nan


## 3. 按月断点重建三股票池共享原始数据


In [ ]:
def require_joinquant_api():
    try:
        get_trade_days(end_date="2019-01-02", count=1)
    except NameError:
        raise RuntimeError("REBUILD_DATA=True requires JoinQuant research runtime")
    except Exception:
        pass


def month_first_trade_dates(start_date, end_date):
    days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    out = []
    last_key = None
    for value in days:
        key = value.strftime("%Y-%m")
        if key != last_key:
            out.append(value.strftime("%Y-%m-%d"))
            last_key = key
    return out


def previous_trade_date(date_value):
    days = pd.to_datetime(get_trade_days(end_date=date_value, count=2))
    return days[-2].strftime("%Y-%m-%d") if len(days) >= 2 else None


def base_eligible_a_shares(feature_date):
    securities = get_all_securities(types=["stock"], date=feature_date)
    if securities is None or securities.empty:
        return []
    cutoff = pd.Timestamp(feature_date).to_pydatetime().date() - _dt.timedelta(days=MIN_LISTING_DAYS)
    stocks = []
    for stock, row in securities.iterrows():
        if not (str(stock).endswith(".XSHG") or str(stock).endswith(".XSHE")):
            continue
        display_name = str(row.get("display_name", ""))
        if "退" in display_name:
            continue
        start_date = row.get("start_date", None)
        if pd.isnull(start_date) or pd.Timestamp(start_date).date() > cutoff:
            continue
        stocks.append(stock)
    try:
        st_df = get_extras("is_st", stocks, count=1, end_date=feature_date)
    except Exception:
        st_df = None
    if st_df is not None and len(st_df):
        st_row = st_df.iloc[-1]
        stocks = [s for s in stocks if s not in st_row.index or pd.isnull(st_row[s]) or not bool(st_row[s])]
    return stocks


def scan_liquidity(stock_list, feature_date):
    rows = []
    stock_chunks = list(chunks(stock_list, LIQUIDITY_SCAN_CHUNK_SIZE))
    for stock_chunk in progress_iter(stock_chunks, total=len(stock_chunks), desc="scan all-A liquidity", leave=False):
        try:
            price = get_price(stock_chunk, end_date=feature_date, frequency="daily", fields=["money", "paused"],
                              count=20, skip_paused=False, panel=False, fill_paused=True)
        except Exception:
            price = None
        if price is None or price.empty:
            continue
        if "code" not in price.columns:
            continue
        for stock, part in price.groupby("code"):
            money = pd.to_numeric(part["money"], errors="coerce") if "money" in part.columns else pd.Series(dtype=float)
            paused = pd.to_numeric(part["paused"], errors="coerce") if "paused" in part.columns else pd.Series(dtype=float)
            rows.append({
                "stock": stock,
                "avg_money_20": float(money.mean()) if len(money.dropna()) else np.nan,
                "paused_last": bool(paused.iloc[-1]) if len(paused.dropna()) else False,
            })
        del price
        gc.collect()
    return pd.DataFrame(rows)


def fetch_jqfactors(stock_list, feature_date):
    out = pd.DataFrame(index=stock_list)
    factor_chunks = list(chunks(FETCH_JQFACTORS, FACTOR_CHUNK_SIZE))
    for factor_chunk in progress_iter(factor_chunks, total=len(factor_chunks), desc="fetch jqfactors", leave=False):
        try:
            values = get_factor_values(securities=stock_list, factors=factor_chunk, count=1, end_date=feature_date)
        except Exception:
            values = None
        for factor in factor_chunk:
            try:
                out[factor] = values[factor].iloc[-1].reindex(stock_list) if values is not None and factor in values else np.nan
            except Exception:
                out[factor] = np.nan
    return out


def fetch_v46_price_features(stock_list, feature_date):
    columns = ["liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60"]
    outputs = []
    stock_chunks = list(chunks(stock_list, PRICE_CHUNK_SIZE))
    for stock_chunk in progress_iter(stock_chunks, total=len(stock_chunks), desc="fetch V46 price path", leave=False):
        out = pd.DataFrame(index=stock_chunk, columns=columns, dtype=float)
        try:
            price = get_price(stock_chunk, end_date=feature_date, frequency="daily",
                              fields=["close", "money", "paused"], count=61, skip_paused=False,
                              fq="pre", panel=False, fill_paused=True)
        except Exception:
            price = None
        if price is not None and not price.empty and "code" in price.columns:
            price["time"] = pd.to_datetime(price["time"]).dt.normalize()
            close = price.pivot_table(index="time", columns="code", values="close").sort_index()
            money = price.pivot_table(index="time", columns="code", values="money").sort_index()
            paused = price.pivot_table(index="time", columns="code", values="paused").sort_index()
            last_close = close.iloc[-1] if len(close) else pd.Series(dtype=float)
            out["px_close_to_ma60"] = last_close / close.tail(60).mean() - 1
            out["px_drawdown_60"] = last_close / close.tail(60).max() - 1
            out["liq_money_ratio_20_60"] = money.tail(20).mean() / money.tail(60).mean() - 1
            out["liq_paused_count_20"] = paused.tail(20).fillna(0).sum()
        outputs.append(out.replace([np.inf, -np.inf], np.nan))
        del out, price
        gc.collect()
    return pd.concat(outputs).reindex(stock_list) if len(outputs) else pd.DataFrame(index=stock_list, columns=columns)


def fetch_forward_returns(stock_list, rebalance_date, next_date):
    pieces = []
    stock_chunks = list(chunks(stock_list, PRICE_CHUNK_SIZE))
    for stock_chunk in progress_iter(stock_chunks, total=len(stock_chunks), desc="fetch forward returns", leave=False):
        try:
            price = get_price(stock_chunk, start_date=rebalance_date, end_date=next_date,
                              frequency="daily", fields=["close"], skip_paused=True, fq="pre", panel=False)
        except Exception:
            price = None
        if price is None or price.empty or "code" not in price.columns:
            continue
        price["time"] = pd.to_datetime(price["time"]).dt.normalize()
        close = price.pivot_table(index="time", columns="code", values="close").sort_index()
        if len(close) >= 2:
            first_pos = 1 if len(close) >= 3 else 0
            pieces.append((close.iloc[-1] / close.iloc[first_pos] - 1).replace([np.inf, -np.inf], np.nan))
        del price, close
        gc.collect()
    return pd.concat(pieces) if len(pieces) else pd.Series(dtype=float)


def fetch_benchmark_return(rebalance_date, next_date):
    try:
        price = get_price(BENCHMARK, start_date=rebalance_date, end_date=next_date,
                          frequency="daily", fields=["close"], skip_paused=True, fq="pre")
    except Exception:
        price = None
    if price is None or price.empty or len(price) < 2:
        return np.nan
    first_pos = 1 if len(price) >= 3 else 0
    return float(price["close"].iloc[-1] / price["close"].iloc[first_pos] - 1)


def fetch_industry_map(stock_list, feature_date):
    try:
        values = get_industry(stock_list, date=feature_date)
    except Exception:
        return dict((s, "UNKNOWN") for s in stock_list)
    out = {}
    for stock in stock_list:
        info = values.get(stock, {}) if values is not None else {}
        sub = info.get("sw_l1", None)
        out[stock] = sub.get("industry_code", "UNKNOWN") if isinstance(sub, dict) else "UNKNOWN"
    return out


In [ ]:
def build_one_month(rebalance_date, next_date):
    feature_date = previous_trade_date(rebalance_date)
    if feature_date is None:
        return pd.DataFrame()
    eligible = base_eligible_a_shares(feature_date)
    liquidity = scan_liquidity(eligible, feature_date)
    liquidity = liquidity[(~liquidity["paused_last"]) & liquidity["avg_money_20"].notnull()].copy()
    liquidity = liquidity.sort_values("avg_money_20", ascending=False)
    liquid_set = set(liquidity.head(LIQUID_ALL_A_SIZE)["stock"].tolist())
    tradable_set = set(liquidity["stock"].tolist())
    hs300_set = set(get_index_stocks("000300.XSHG", feature_date)).intersection(tradable_set)
    csi800_set = set(get_index_stocks("000906.XSHG", feature_date)).intersection(tradable_set)
    union = _bi.sorted(hs300_set.union(csi800_set).union(liquid_set))
    if len(union) == 0:
        return pd.DataFrame()

    factors = fetch_jqfactors(union, feature_date)
    price_features = fetch_v46_price_features(union, feature_date)
    returns = fetch_forward_returns(union, rebalance_date, next_date)
    benchmark_return = fetch_benchmark_return(rebalance_date, next_date)
    industry_map = fetch_industry_map(union, feature_date)
    liquidity_map = liquidity.set_index("stock")["avg_money_20"] if len(liquidity) else pd.Series(dtype=float)

    month = factors.join(price_features, how="left")
    month[STOCK_COL] = month.index.astype(str)
    month["raw_return_1m"] = returns.reindex(month.index)
    month[TARGET_COL] = month["raw_return_1m"] - benchmark_return
    month["benchmark_csi800_1m"] = benchmark_return
    month["audit_avg_money_20"] = month[STOCK_COL].map(liquidity_map)
    month[INDUSTRY_COL] = month[STOCK_COL].map(industry_map).fillna("UNKNOWN")
    month["in_hs300"] = month[STOCK_COL].isin(hs300_set)
    month["in_csi800"] = month[STOCK_COL].isin(csi800_set)
    month["in_liquid_all_a_1500"] = month[STOCK_COL].isin(liquid_set)
    month[DATE_COL] = rebalance_date
    month["feature_date"] = feature_date
    month[NEXT_DATE_COL] = next_date
    month = month.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    return month


def add_temporal_v46_features(universe_df):
    out = universe_df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    for factor in ["cash_flow_to_price_ratio", "Rank1M"]:
        values = pd.to_numeric(out[factor], errors="coerce").replace([np.inf, -np.inf], np.nan)
        rank_col = "_tmp_rank_" + factor
        out[rank_col] = values.groupby(out[DATE_COL]).rank(method="average", pct=True)
        grouped = out.groupby(STOCK_COL)[rank_col]
        if factor == "cash_flow_to_price_ratio":
            out["ts_cash_flow_to_price_ratio_rank_mean_3m"] = grouped.transform(
                lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        else:
            out["ts_Rank1M_rank_chg_1m"] = out[rank_col] - grouped.shift(1)
        out = out.drop(columns=[rank_col])
    return out


def coerce_membership(values):
    if str(values.dtype) == "bool":
        return values
    return values.astype(str).str.lower().isin(["true", "1", "yes"])


def build_or_load_panel():
    if DATA_PATH_OVERRIDE:
        path = Path(DATA_PATH_OVERRIDE)
        if not path.exists():
            raise IOError("DATA_PATH_OVERRIDE not found: %s" % path)
        return pd.read_csv(path), path
    if PANEL_PATH.exists() and not REBUILD_DATA:
        return pd.read_csv(PANEL_PATH), PANEL_PATH
    if not REBUILD_DATA:
        raise IOError("panel not found; set REBUILD_DATA=True")
    require_joinquant_api()
    dates = month_first_trade_dates(REBUILD_START, REBUILD_END_RESOLVED)
    tasks = list(enumerate(dates[:-1]))
    for i, rebalance_date in progress_iter(tasks, total=len(tasks), desc="rebuild V88 monthly cache"):
        next_date = dates[i + 1]
        cache_path = CACHE_DIR / ("v88_raw_%s.csv" % rebalance_date.replace("-", ""))
        if cache_path.exists() and not REBUILD_FORCE_MONTHS:
            continue
        month = build_one_month(rebalance_date, next_date)
        if len(month):
            month.to_csv(cache_path, index=False)
            print("cached", rebalance_date, "rows", len(month))
        del month
        gc.collect()
    cache_files = _bi.sorted(CACHE_DIR.glob("v88_raw_*.csv"))
    if len(cache_files) == 0:
        raise ValueError("no V88 monthly cache generated")
    raw_parts = []
    for path in progress_iter(cache_files, total=len(cache_files), desc="load V88 monthly cache"):
        raw_parts.append(pd.read_csv(path))
    raw = pd.concat(raw_parts, ignore_index=True)
    long_parts = []
    for spec in progress_iter(UNIVERSES, total=len(UNIVERSES), desc="build universe panels"):
        membership = coerce_membership(raw[spec["membership_col"]])
        part = raw[membership].copy()
        part["universe"] = spec["universe"]
        part = add_temporal_v46_features(part)
        long_parts.append(part)
    panel = pd.concat(long_parts, ignore_index=True)
    keep = unique_keep_order([
        STOCK_COL, DATE_COL, "feature_date", NEXT_DATE_COL, "universe", TARGET_COL,
        "raw_return_1m", "benchmark_csi800_1m", INDUSTRY_COL, "audit_avg_money_20", "size"
    ] + FULL_V46_COLS)
    panel = panel[[c for c in keep if c in panel.columns]].copy()
    panel.to_csv(PANEL_PATH, index=False)
    print("saved final panel", PANEL_PATH, panel.shape)
    return panel, PANEL_PATH


df_all, DATA_PATH = build_or_load_panel()
for col in [DATE_COL, "feature_date", NEXT_DATE_COL]:
    df_all[col] = pd.to_datetime(df_all[col], errors="coerce").dt.normalize()
df_all[STOCK_COL] = df_all[STOCK_COL].astype(str)
for col in progress_iter(FULL_V46_COLS + [TARGET_COL, "audit_avg_money_20", "size"],
                         total=len(FULL_V46_COLS) + 3, desc="compact V88 panel"):
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce").astype(np.float32)
missing_features = [c for c in FULL_V46_COLS if c not in df_all.columns]
if missing_features:
    raise ValueError("V88 panel missing V46 features: %s" % missing_features)
df_all = df_all.dropna(subset=[STOCK_COL, DATE_COL, NEXT_DATE_COL, TARGET_COL, "universe"])
print("DATA_PATH:", DATA_PATH)
print("panel:", df_all.shape, "months:", df_all[DATE_COL].nunique())
display_df(df_all.groupby("universe").size().reset_index(name="rows"), 10)


## 4. 因子适配、覆盖率与股票池画像审计


In [ ]:
universe_month_size_df = df_all.groupby(["universe", DATE_COL]).size().reset_index(name="stock_count")
universe_month_size_df.to_csv(OUT_DIR / "v88_universe_month_size.csv", index=False)

factor_audit_rows = []
for universe, universe_df in progress_iter(list(df_all.groupby("universe")), total=df_all["universe"].nunique(), desc="audit universe factors"):
    for factor in progress_iter(FULL_V46_COLS, total=len(FULL_V46_COLS), desc="factor coverage %s" % universe, leave=False):
        values = pd.to_numeric(universe_df[factor], errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = values.dropna()
        monthly_std = pd.DataFrame({DATE_COL: universe_df[DATE_COL], "_value": values}).groupby(DATE_COL)["_value"].std()
        factor_audit_rows.append({
            "universe": universe, "factor": factor, "rows": int(len(values)),
            "coverage": float(values.notnull().mean()),
            "median": float(valid.median()) if len(valid) else np.nan,
            "p01": float(valid.quantile(0.01)) if len(valid) else np.nan,
            "p99": float(valid.quantile(0.99)) if len(valid) else np.nan,
            "iqr": float(valid.quantile(0.75) - valid.quantile(0.25)) if len(valid) else np.nan,
            "zero_rate": float((valid == 0).mean()) if len(valid) else np.nan,
            "mean_cross_section_std": safe_mean(monthly_std),
        })
factor_audit_df = pd.DataFrame(factor_audit_rows)

baseline_audit = factor_audit_df[factor_audit_df["universe"] == BASELINE_UNIVERSE][
    ["factor", "coverage", "median", "iqr"]].copy()
baseline_audit = baseline_audit.rename(columns={"coverage": "baseline_coverage", "median": "baseline_median", "iqr": "baseline_iqr"})
factor_drift_df = factor_audit_df.merge(baseline_audit, on="factor", how="left")
factor_drift_df["coverage_delta_vs_csi800"] = factor_drift_df["coverage"] - factor_drift_df["baseline_coverage"]
denom = factor_drift_df["baseline_iqr"].abs().where(factor_drift_df["baseline_iqr"].abs() > 1e-12)
factor_drift_df["median_shift_in_csi800_iqr"] = (factor_drift_df["median"] - factor_drift_df["baseline_median"]) / denom
factor_drift_df["adaptation_flag"] = (
    (factor_drift_df["coverage"] < 0.80)
    | (factor_drift_df["coverage_delta_vs_csi800"] < -0.10)
    | (factor_drift_df["median_shift_in_csi800_iqr"].abs() > 1.0)
)

factor_audit_df.to_csv(OUT_DIR / "v88_factor_applicability.csv", index=False)
factor_drift_df.to_csv(OUT_DIR / "v88_factor_distribution_drift.csv", index=False)
print("factor adaptation flags")
display_df(factor_drift_df[factor_drift_df["adaptation_flag"]].sort_values(
    ["universe", "coverage"])[["universe", "factor", "coverage", "coverage_delta_vs_csi800", "median_shift_in_csi800_iqr"]], 60)


## 5. Walk-forward训练与月度排名评估


In [ ]:
def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            value = corr.iloc[i, j]
            if not pd.isnull(value) and abs(value) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    visited = set()
    components = []
    for col in feature_cols:
        if col in visited:
            continue
        stack = [col]
        component = []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(graph[current])
        components.append(component)
    return components


def select_features_train_only(train_df):
    missing = train_df[FULL_V46_COLS].isnull().sum().to_dict()
    keep = []
    remove = []
    for component in build_corr_components(train_df, FULL_V46_COLS, CORR_THRESHOLD):
        ordered = _bi.sorted(component, key=lambda x: (missing[x], x))
        keep.append(ordered[0])
        remove.extend(ordered[1:])
    return keep, remove


def model_params(seed):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = int(seed)
    params["feature_fraction_seed"] = int(seed)
    params["bagging_seed"] = int(seed)
    params["data_random_seed"] = int(seed)
    params["num_threads"] = int(NUM_THREADS)
    return params


def make_fold_frames(universe_df, spec):
    cutoff = pd.Timestamp(spec["cutoff"])
    test_start = pd.Timestamp(spec["test_start"])
    test_end = pd.Timestamp(spec["test_end"])
    train = universe_df[(universe_df[DATE_COL] <= cutoff) & (universe_df[NEXT_DATE_COL] <= cutoff)].copy()
    test = universe_df[(universe_df[DATE_COL] >= test_start) & (universe_df[DATE_COL] <= test_end)].copy()
    return train, test


def evaluate_months(test_meta, predictions, universe, fold_id, seed):
    panel = test_meta.copy()
    panel["score"] = np.asarray(predictions, dtype=float)
    metric_rows = []
    selected_rows = []
    groups = panel.groupby(DATE_COL)
    for rebalance_date, month_df in progress_iter(groups, total=panel[DATE_COL].nunique(),
                                                   desc="evaluate %s %s s%s" % (universe, fold_id, seed), leave=False):
        month_df = month_df.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET_COL, "score"]).copy()
        n = int(len(month_df))
        if n < 30:
            continue
        true_n = _bi.max(1, int(round(n * TRUE_TOP_FRACTION)))
        true_sorted = month_df.sort_values(TARGET_COL, ascending=False)
        true_index = set(true_sorted.head(true_n).index)
        scored = month_df.sort_values("score", ascending=False)
        selected_index = board_capped_indices(scored, PRED_TOP_K)
        selected = month_df.loc[selected_index].copy()
        hits = int(len(true_index.intersection(set(selected_index))))
        binary_true = month_df.index.to_series().isin(true_index).astype(int).values
        precision = hits / float(len(selected_index)) if len(selected_index) else np.nan
        random_precision = true_n / float(n)
        top8_alpha = float(selected[TARGET_COL].mean()) if len(selected) else np.nan
        universe_alpha = float(month_df[TARGET_COL].mean())
        metric_rows.append({
            "universe": universe, "fold_id": fold_id, "seed": int(seed),
            "rebalance_date": pd.Timestamp(rebalance_date), "n": n, "true_top_n": true_n,
            "rank_ic": safe_rank_ic(month_df[TARGET_COL], month_df["score"]),
            "auc_true_top_pct": binary_rank_auc(binary_true, month_df["score"].values),
            "precision_at8_true_top_pct": precision,
            "precision_lift_at8": precision / random_precision if random_precision > 0 else np.nan,
            "recall_at8_true_top_pct": hits / float(true_n),
            "ndcg_at8_true_top_pct": binary_ndcg_at_k(binary_true, month_df["score"].values, PRED_TOP_K),
            "top8_alpha": top8_alpha,
            "top8_edge": top8_alpha - universe_alpha if not pd.isnull(top8_alpha) else np.nan,
        })
        true_rank = month_df[TARGET_COL].rank(method="first", ascending=False)
        score_rank = month_df["score"].rank(method="first", ascending=False)
        for idx in selected_index:
            row = month_df.loc[idx]
            selected_rows.append({
                "universe": universe, "fold_id": fold_id, "seed": int(seed),
                "rebalance_date": pd.Timestamp(rebalance_date), "stock": row[STOCK_COL],
                "score_rank": int(score_rank.loc[idx]), "true_rank": int(true_rank.loc[idx]),
                "is_true_top_pct": bool(idx in true_index), "alpha_1m": float(row[TARGET_COL]),
                "board": stock_board(row[STOCK_COL]),
                "industry_bucket": row.get(INDUSTRY_COL, "UNKNOWN"),
                "audit_size": row.get("size", np.nan),
                "audit_avg_money_20": row.get("audit_avg_money_20", np.nan),
            })
    return metric_rows, selected_rows


In [ ]:
fold_plan_rows = []
monthly_rows = []
selected_rows = []
model_meta_rows = []
importance_rows = []

for universe_spec in progress_iter(UNIVERSES, total=len(UNIVERSES), desc="V88 universes"):
    universe = universe_spec["universe"]
    universe_df = df_all[df_all["universe"] == universe].copy()
    for fold_spec in progress_iter(FOLD_SPECS, total=len(FOLD_SPECS), desc="folds %s" % universe, leave=False):
        train_df, test_df = make_fold_frames(universe_df, fold_spec)
        train_months = int(train_df[DATE_COL].nunique())
        test_months = int(test_df[DATE_COL].nunique())
        status = "run" if train_months >= MIN_TRAIN_MONTHS and test_months > 0 else "skip"
        fold_plan_rows.append({
            "universe": universe, "fold_id": fold_spec["fold_id"], "cutoff": fold_spec["cutoff"],
            "train_rows": int(len(train_df)), "train_months": train_months,
            "test_rows": int(len(test_df)), "test_months": test_months, "status": status,
        })
        if status != "run":
            del train_df, test_df
            gc.collect()
            continue
        feature_cols, removed_cols = select_features_train_only(train_df)
        fill_values = train_df[feature_cols].median().replace([np.inf, -np.inf], np.nan).fillna(0)
        X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
        y_train = train_df[TARGET_COL].astype(float)
        X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
        meta_cols = [DATE_COL, STOCK_COL, TARGET_COL, INDUSTRY_COL, "audit_avg_money_20", "size"]
        test_meta = test_df[[c for c in meta_cols if c in test_df.columns]].copy().reset_index(drop=True)
        X_test = X_test.reset_index(drop=True)
        dtrain = lgb.Dataset(X_train, label=y_train, feature_name=list(feature_cols), free_raw_data=False)
        if hasattr(dtrain, "construct"):
            dtrain.construct()

        for seed in progress_iter(SEEDS, total=len(SEEDS), desc="seeds %s %s" % (universe, fold_spec["fold_id"]), leave=False):
            model = lgb.train(model_params(seed), dtrain, num_boost_round=int(FIXED_ITER))
            predictions = np.asarray(model.predict(X_test[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
            metric_one, selected_one = evaluate_months(test_meta, predictions, universe, fold_spec["fold_id"], seed)
            monthly_rows.extend(metric_one)
            selected_rows.extend(selected_one)
            gain = np.asarray(model.feature_importance(importance_type="gain"), dtype=float)
            gain_total = float(gain.sum())
            for i, feature in enumerate(feature_cols):
                importance_rows.append({
                    "universe": universe, "fold_id": fold_spec["fold_id"], "seed": int(seed),
                    "feature": feature, "importance_gain_pct": float(gain[i] / gain_total) if gain_total > 0 else np.nan,
                })
            model_meta_rows.append({
                "universe": universe, "fold_id": fold_spec["fold_id"], "seed": int(seed),
                "cutoff": fold_spec["cutoff"], "train_rows": int(len(train_df)), "train_months": train_months,
                "test_rows": int(len(test_df)), "test_months": test_months,
                "feature_count": int(len(feature_cols)), "removed_features": ",".join(removed_cols),
            })
            del model, predictions
            gc.collect()
        del dtrain, X_train, X_test, y_train, test_meta, train_df, test_df
        gc.collect()
    del universe_df
    gc.collect()

fold_plan_df = pd.DataFrame(fold_plan_rows)
monthly_metrics_df = pd.DataFrame(monthly_rows)
selected_detail_df = pd.DataFrame(selected_rows)
model_meta_df = pd.DataFrame(model_meta_rows)
feature_importance_df = pd.DataFrame(importance_rows)

for filename, frame in progress_iter([
    ("v88_fold_plan.csv", fold_plan_df), ("v88_monthly_metrics.csv", monthly_metrics_df),
    ("v88_selected_detail.csv", selected_detail_df), ("v88_model_meta.csv", model_meta_df),
    ("v88_feature_importance.csv", feature_importance_df),
], total=5, desc="save V88 raw outputs"):
    frame.to_csv(OUT_DIR / filename, index=False)
print("models:", len(model_meta_df), "monthly metrics:", len(monthly_metrics_df))
display_df(model_meta_df, 20)


## 6. 去重月度汇总、配对比较与判定


In [ ]:
METRIC_COLS = [
    "rank_ic", "auc_true_top_pct", "precision_at8_true_top_pct", "precision_lift_at8",
    "recall_at8_true_top_pct", "ndcg_at8_true_top_pct", "top8_alpha", "top8_edge",
]


def summarize_frame(df, group_cols):
    rows = []
    grouped = list(df.groupby(group_cols)) if len(group_cols) else [((), df)]
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize V88", leave=False):
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)
        row = dict((group_cols[i], keys[i]) for i in range(len(group_cols)))
        row["months"] = int(len(part))
        for metric in METRIC_COLS:
            values = pd.to_numeric(part[metric], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            row[metric + "_mean"] = float(values.mean()) if len(values) else np.nan
            row[metric + "_median"] = float(values.median()) if len(values) else np.nan
            threshold = 0.5 if metric == "auc_true_top_pct" else (1.0 if metric == "precision_lift_at8" else 0.0)
            row[metric + "_win_rate"] = float((values > threshold).mean()) if len(values) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


# Average stochastic seeds first so the same calendar month is not treated as three independent observations.
unique_month_df = monthly_metrics_df.groupby(["universe", DATE_COL])[METRIC_COLS].mean().reset_index()
universe_summary_df = summarize_frame(unique_month_df, ["universe"])
seed_summary_df = summarize_frame(monthly_metrics_df, ["universe", "seed"])
fold_summary_df = summarize_frame(monthly_metrics_df, ["universe", "fold_id"])


def pair_vs_csi800(candidate):
    keys = ["fold_id", "seed", DATE_COL]
    base = monthly_metrics_df[monthly_metrics_df["universe"] == BASELINE_UNIVERSE][keys + METRIC_COLS].copy()
    challenger = monthly_metrics_df[monthly_metrics_df["universe"] == candidate][keys + METRIC_COLS].copy()
    base = base.rename(columns=dict((c, c + "_csi800") for c in METRIC_COLS))
    challenger = challenger.rename(columns=dict((c, c + "_candidate") for c in METRIC_COLS))
    paired = base.merge(challenger, on=keys, how="inner")
    paired["candidate_universe"] = candidate
    for metric in METRIC_COLS:
        paired[metric + "_delta"] = paired[metric + "_candidate"] - paired[metric + "_csi800"]
    return paired


pairwise_parts = []
candidate_universes = [x for x in UNIVERSE_NAMES if x != BASELINE_UNIVERSE]
for candidate in progress_iter(candidate_universes, total=len(candidate_universes), desc="pair universes vs CSI800"):
    pairwise_parts.append(pair_vs_csi800(candidate))
pairwise_df = pd.concat(pairwise_parts, ignore_index=True)


def summarize_deltas(df, group_cols):
    rows = []
    grouped = list(df.groupby(group_cols)) if len(group_cols) else [((), df)]
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize universe deltas", leave=False):
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)
        row = dict((group_cols[i], keys[i]) for i in range(len(group_cols)))
        row["paired_month_seed_rows"] = int(len(part))
        for metric in METRIC_COLS:
            values = pd.to_numeric(part[metric + "_delta"], errors="coerce").dropna()
            row[metric + "_delta_mean"] = float(values.mean()) if len(values) else np.nan
            row[metric + "_delta_positive_rate"] = float((values > 0).mean()) if len(values) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


pairwise_overall_df = summarize_deltas(pairwise_df, ["candidate_universe"])
pairwise_seed_df = summarize_deltas(pairwise_df, ["candidate_universe", "seed"])
pairwise_fold_df = summarize_deltas(pairwise_df, ["candidate_universe", "fold_id"])

decision_rows = []
for candidate in progress_iter(candidate_universes, total=len(candidate_universes), desc="build universe decision"):
    overall = pairwise_overall_df[pairwise_overall_df["candidate_universe"] == candidate].iloc[0]
    seed_part = pairwise_seed_df[pairwise_seed_df["candidate_universe"] == candidate]
    fold_part = pairwise_fold_df[pairwise_fold_df["candidate_universe"] == candidate]
    candidate_level = universe_summary_df[universe_summary_df["universe"] == candidate].iloc[0]
    seed_edge_wins = int((seed_part["top8_edge_delta_mean"] > 0).sum())
    fold_edge_wins = int((fold_part["top8_edge_delta_mean"] > 0).sum())
    rank_not_materially_worse = bool(overall["rank_ic_delta_mean"] >= -0.005)
    lift_usable = bool(candidate_level["precision_lift_at8_mean"] >= 1.5)
    edge_improves = bool(overall["top8_edge_delta_mean"] > 0)
    robust_edge = bool(seed_edge_wins >= 2 and fold_edge_wins >= 3)
    helpful = bool(rank_not_materially_worse and lift_usable and edge_improves and robust_edge)
    decision_rows.append({
        "candidate_universe": candidate,
        "decision": "opportunity_set_helpful" if helpful else "keep_csi800_mainline",
        "rank_ic_delta_mean": overall["rank_ic_delta_mean"],
        "precision_lift_level": candidate_level["precision_lift_at8_mean"],
        "precision_lift_delta_mean": overall["precision_lift_at8_delta_mean"],
        "top8_edge_delta_mean": overall["top8_edge_delta_mean"],
        "seed_edge_wins": seed_edge_wins, "seed_total": int(len(seed_part)),
        "fold_edge_wins": fold_edge_wins, "fold_total": int(len(fold_part)),
        "rank_not_materially_worse": rank_not_materially_worse,
        "precision_lift_ge_1_5": lift_usable, "edge_improves": edge_improves,
        "robust_edge": robust_edge,
    })
decision_df = pd.DataFrame(decision_rows)

# Seed agreement is a direct audit of Top8 path stability.
agreement_rows = []
for keys, part in progress_iter(list(selected_detail_df.groupby(["universe", DATE_COL])),
                                total=selected_detail_df.groupby(["universe", DATE_COL]).ngroups,
                                desc="audit seed agreement"):
    universe, date_value = keys
    seed_sets = dict((seed, set(group[STOCK_COL].tolist())) for seed, group in part.groupby("seed"))
    seeds_here = _bi.sorted(seed_sets.keys())
    overlaps = []
    for i in range(len(seeds_here)):
        for j in range(i + 1, len(seeds_here)):
            overlaps.append(len(seed_sets[seeds_here[i]].intersection(seed_sets[seeds_here[j]])) / float(PRED_TOP_K))
    agreement_rows.append({"universe": universe, DATE_COL: date_value,
                           "mean_pairwise_top8_overlap": safe_mean(overlaps), "seed_count": len(seeds_here)})
seed_agreement_df = pd.DataFrame(agreement_rows)

outputs = {
    "v88_unique_month_metrics.csv": unique_month_df,
    "v88_universe_summary.csv": universe_summary_df,
    "v88_seed_summary.csv": seed_summary_df,
    "v88_fold_summary.csv": fold_summary_df,
    "v88_pairwise_vs_csi800_monthly.csv": pairwise_df,
    "v88_pairwise_vs_csi800_overall.csv": pairwise_overall_df,
    "v88_pairwise_vs_csi800_by_seed.csv": pairwise_seed_df,
    "v88_pairwise_vs_csi800_by_fold.csv": pairwise_fold_df,
    "v88_universe_decision_table.csv": decision_df,
    "v88_seed_agreement_monthly.csv": seed_agreement_df,
}
for filename, frame in progress_iter(list(outputs.items()), total=len(outputs), desc="save V88 summaries"):
    frame.to_csv(OUT_DIR / filename, index=False)

print("universe decision")
display_df(decision_df, 10)
display_df(universe_summary_df, 10)


## 7. 选股画像与可视化


In [ ]:
trait_rows = []
for universe, part in progress_iter(list(selected_detail_df.groupby("universe")),
                                    total=selected_detail_df["universe"].nunique(), desc="selected traits"):
    size_values = pd.to_numeric(part["audit_size"], errors="coerce").dropna()
    money_values = pd.to_numeric(part["audit_avg_money_20"], errors="coerce").dropna()
    trait_rows.append({
        "universe": universe, "selected_rows": int(len(part)),
        "size_median": float(size_values.median()) if len(size_values) else np.nan,
        "avg_money_20_median": float(money_values.median()) if len(money_values) else np.nan,
        "chinext_rate": float((part["board"] == "chinext").mean()),
        "star_rate": float((part["board"] == "star").mean()),
        "industry_count": int(part["industry_bucket"].nunique()),
        "top_industry_rate": float(part["industry_bucket"].value_counts(normalize=True).iloc[0]) if len(part) else np.nan,
    })
selected_trait_df = pd.DataFrame(trait_rows)
selected_trait_df.to_csv(OUT_DIR / "v88_selected_trait_summary.csv", index=False)

industry_profile_df = selected_detail_df.groupby(["universe", "industry_bucket"]).size().reset_index(name="selected_count")
industry_total = industry_profile_df.groupby("universe")["selected_count"].transform("sum")
industry_profile_df["selected_rate"] = industry_profile_df["selected_count"] / industry_total
industry_profile_df.to_csv(OUT_DIR / "v88_selected_industry_profile.csv", index=False)


def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(str(path), dpi=140, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print("saved figure:", path)


colors = ["#59636d", "#2d8f78", "#d4543c"]
summary_plot = universe_summary_df.set_index("universe").reindex(UNIVERSE_NAMES)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
plot_specs = [
    ("rank_ic_mean", "Mean OOS RankIC", 0.0),
    ("precision_lift_at8_mean", "Precision Lift@8", 1.0),
    ("top8_edge_mean", "Mean Top8 edge", 0.0),
    ("auc_true_top_pct_mean", "AUC true Top2.5%", 0.5),
]
for i, spec in enumerate(plot_specs):
    ax = axes.ravel()[i]
    ax.bar(np.arange(len(UNIVERSE_NAMES)), summary_plot[spec[0]].values, color=colors)
    ax.axhline(spec[2], color="#333333", linewidth=0.8)
    ax.set_xticks(np.arange(len(UNIVERSE_NAMES)))
    ax.set_xticklabels(UNIVERSE_NAMES, rotation=20, ha="right")
    ax.set_title(spec[1])
save_figure(fig, "v88_universe_metric_dashboard.png")

folds = [x["fold_id"] for x in FOLD_SPECS]
positions = np.arange(len(folds))
width = 0.24
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
for i, universe in progress_iter(enumerate(UNIVERSE_NAMES), total=len(UNIVERSE_NAMES), desc="plot fold comparison", leave=False):
    part = fold_summary_df[fold_summary_df["universe"] == universe].set_index("fold_id").reindex(folds)
    offset = (i - 1) * width
    axes[0].bar(positions + offset, part["rank_ic_mean"].values, width=width, color=colors[i], label=universe)
    axes[1].bar(positions + offset, part["top8_edge_mean"].values, width=width, color=colors[i], label=universe)
axes[0].axhline(0, color="#333333", linewidth=0.8)
axes[1].axhline(0, color="#333333", linewidth=0.8)
axes[0].set_title("RankIC by OOS fold")
axes[1].set_title("Top8 edge by OOS fold")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(folds)
axes[0].legend(loc="best")
axes[1].legend(loc="best")
save_figure(fig, "v88_fold_stability.png")

all_a_drift = factor_drift_df[factor_drift_df["universe"] == "liquid_all_a_1500"].copy()
all_a_drift["abs_shift"] = all_a_drift["median_shift_in_csi800_iqr"].abs()
all_a_drift = all_a_drift.sort_values("abs_shift", ascending=False).head(20).sort_values("abs_shift")
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(np.arange(len(all_a_drift)), all_a_drift["median_shift_in_csi800_iqr"].values, color="#d4543c")
axes[0].set_yticks(np.arange(len(all_a_drift)))
axes[0].set_yticklabels(all_a_drift["factor"], fontsize=9)
axes[0].set_title("Liquid All-A median shift in CSI800 IQR")
coverage_plot = factor_drift_df.pivot_table(index="factor", columns="universe", values="coverage", aggfunc="last")
coverage_plot = coverage_plot.reindex(FULL_V46_COLS)
for i, universe in enumerate(UNIVERSE_NAMES):
    axes[1].plot(np.arange(len(coverage_plot)), coverage_plot[universe].values, label=universe, color=colors[i])
axes[1].axhline(0.8, color="#333333", linewidth=0.8)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("V46 feature coverage")
axes[1].legend(loc="best")
save_figure(fig, "v88_factor_applicability.png")

agreement_plot = seed_agreement_df.groupby("universe")["mean_pairwise_top8_overlap"].mean().reindex(UNIVERSE_NAMES)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].bar(np.arange(len(UNIVERSE_NAMES)), agreement_plot.values, color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_xticks(np.arange(len(UNIVERSE_NAMES)))
axes[0].set_xticklabels(UNIVERSE_NAMES, rotation=20, ha="right")
axes[0].set_title("Mean pairwise seed Top8 overlap")
trait_plot = selected_trait_df.set_index("universe").reindex(UNIVERSE_NAMES)
axes[1].bar(np.arange(len(UNIVERSE_NAMES)), trait_plot["avg_money_20_median"].values / 1e8, color=colors)
axes[1].set_xticks(np.arange(len(UNIVERSE_NAMES)))
axes[1].set_xticklabels(UNIVERSE_NAMES, rotation=20, ha="right")
axes[1].set_title("Selected median 20d average money (100m CNY)")
save_figure(fig, "v88_seed_agreement_and_selected_liquidity.png")


## 8. 输出说明


In [ ]:
pd.DataFrame({"feature": FULL_V46_COLS, "role": "model"}).to_csv(OUT_DIR / "v88_feature_manifest.csv", index=False)
pd.DataFrame(UNIVERSES).to_csv(OUT_DIR / "v88_universe_manifest.csv", index=False)

readme_lines = [
    "V88 V46 universe boundary and Liquid All-A applicability experiment",
    "Run timestamp: %s" % RUN_TIMESTAMP,
    "Panel: %s" % DATA_PATH,
    "Only universe changes; V46 features, raw-L2, fixed120 and board-capped Top8 stay frozen.",
    "True positives are each universe's forward-return Top2.5%.",
    "Primary result: v88_universe_decision_table.csv",
    "Core metrics: v88_universe_summary.csv and v88_pairwise_vs_csi800_by_fold.csv",
    "Feature adaptation: v88_factor_applicability.csv and v88_factor_distribution_drift.csv",
    "Path stability: v88_seed_agreement_monthly.csv",
    "Monthly raw rebuild files are resumable under csi800_ml_v88_universe_monthly_cache.",
    "This is a walk-forward ranking experiment, not an event-driven execution backtest.",
]
with open(OUT_DIR / "v88_README.txt", "w") as f:
    f.write("\n".join(readme_lines))

print("saved outputs:")
for path in _bi.sorted(OUT_DIR.glob("*.csv")):
    print("-", path)
print("figures:")
for path in _bi.sorted(FIGURE_DIR.glob("*.png")):
    print("-", path)
display_df(decision_df, 10)
